<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/PARC2026_pre/blob/main/smolvla_libero_spatial_lora_0804comeback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmolVLA × LIBERO-plus Spatial LoRA Fine-tuning

`lerobot/smolvla_libero_plus`を初期重みとして、
LIBERO-Spatialの10タスクをLoRAで追加学習します。

学習後はLoRAを元モデルへマージし、次の2モデルを
同じLIBERO-plus Spatial環境で比較します。

- 追加学習前のLIBERO-plus重み
- Spatial追加学習後のマージ済みモデル

**既定条件**

- Spatial 10タスク × 各5エピソード
- 3,000 training steps
- 100 stepsごとにlossを表示
- 評価は10タスク × 各3エピソード


## 1. Colabランタイムを確認する

ColabのランタイムをGPUへ変更してから実行してください。

In [1]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/content/lerobot_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを選択してください。")

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA A100-SXM4-40GB


## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入します。

In [2]:
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


run_quiet(["apt-get", "update", "-qq"])
run_quiet(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

System packages ready.


## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
ColabでのLoRA学習に必要な互換性調整もこのセルで適用します。

In [3]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ]
)

smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

LeRobot ready.


## 4. 学習・評価条件を設定する

Spatialの10タスクから各5エピソードを選び、
合計50エピソードで追加学習します。

評価を正式な10エピソード/taskへ近づける場合は、
`EVAL_EPISODES_PER_TASK = 10`へ変更してください。

In [4]:
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

TRAIN_EPISODES_PER_TASK = 5

STEPS = 3000
LOG_FREQ = 100
BATCH_SIZE = 1
LEARNING_RATE = 3e-4
FINAL_LEARNING_RATE = 3e-5
WARMUP_STEPS = 100
LORA_R = 16
LORA_ALPHA = 16
SEED = 42

EVAL_TASK_IDS = list(range(10))
EVAL_EPISODES_PER_TASK = 3
EVAL_SEED = 2026

OUTPUT_DIR = Path(
    "/content/outputs/smolvla_libero_plus_spatial_lora"
)
MERGED_MODEL_DIR = Path(
    "/content/smolvla_libero_plus_spatial_lora_merged"
)
BASELINE_MODEL_DIR = Path(
    "/content/smolvla_libero_plus_baseline"
)

BASE_EVAL_DIR = Path(
    "/content/eval/base"
)
FINETUNED_EVAL_DIR = Path(
    "/content/eval/spatial_lora"
)
COMPARISON_CSV_PATH = Path(
    "/content/libero_spatial_comparison.csv"
)
MERGED_ZIP_PATH = Path(
    "/content/smolvla_libero_plus_spatial_lora_merged.zip"
)

MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

## 5. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [5]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 6. Spatial学習データを選ぶ

10タスクから各5エピソードを等間隔に選択します。

In [6]:
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

selected_by_task: dict[str, list[int]] = {}

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )

    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )

    selected_by_task[actual_task] = choose_evenly_spaced(
        task_to_episodes[actual_task],
        TRAIN_EPISODES_PER_TASK,
    )

EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

if len(EPISODE_INDICES) != 50:
    raise RuntimeError("Episode selection failed.")

print("Training data: 10 tasks × 5 episodes = 50 episodes")

/usr/local/lib/python3.12/dist-packages/datasets/utils/tqdm.py:86: UserWarning: Cannot enable progress bars: environment variable `HF_DATASETS_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


Training data: 10 tasks × 5 episodes = 50 episodes


## 7. 初期重みを準備する

In [7]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "train_config.json",
        "policy_preprocessor.json",
        "policy_preprocessor*.safetensors",
        "policy_postprocessor.json",
        "policy_postprocessor*.safetensors",
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",
    ],
)

if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

Base model ready.


## 8. LoRA学習を実行する

100 stepごとに平均lossとlearning rateを表示します。

In [ ]:
import re
from collections import deque

episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_spatial_lora",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(
            r"loss:([0-9.eE+-]+)",
            line,
        )
        lr_match = re.search(
            r"lr:([0-9.eE+-]+)",
            line,
        )

        loss = (
            loss_match.group(1)
            if loss_match
            else "n/a"
        )
        lr = (
            lr_match.group(1)
            if lr_match
            else "n/a"
        )

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

Preparing data and starting training...
step  100/3000  loss=0.132  lr=1.5e-04
step  200/3000  loss=0.129  lr=3.0e-04
step  300/3000  loss=0.149  lr=3.0e-04
step  400/3000  loss=0.167  lr=2.9e-04
step  500/3000  loss=0.145  lr=2.9e-04
step  600/3000  loss=0.130  lr=2.8e-04
step  700/3000  loss=0.151  lr=2.7e-04
step  800/3000  loss=0.147  lr=2.6e-04


## 9. LoRAをマージしてモデル全体を保存する

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存します。

In [ ]:
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

gc.collect()
torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

## 10. 比較用ベースラインを準備する

公開weightを追加学習モデルと同じ入力schema・processorへ揃えます。

In [ ]:
baseline_config = PreTrainedConfig.from_pretrained(
    MERGED_MODEL_DIR
)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    baseline_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=baseline_config,
        strict=False,
    )

shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(
            source_path,
            BASELINE_MODEL_DIR / source_path.name,
        )

del baseline_policy
gc.collect()
torch.cuda.empty_cache()

print("Baseline ready.")

## 11. LIBERO-plus評価環境を準備する

MuJoCo、LIBERO-plus fork、評価assetsを導入します。

In [11]:
from huggingface_hub import hf_hub_download

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = Path("/content/LIBERO-plus")
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
LIBERO_PLUS_ASSETS_DIR = (
    LIBERO_PLUS_PACKAGE_ROOT / "assets"
)

os.environ["MUJOCO_GL"] = "egl"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
    ]
)

if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = Path(
        "/content/libero_plus_assets"
    )
    archive_path = Path(
        run_hf_with_retry(
            lambda: hf_hub_download(
                repo_id="Sylvest/LIBERO-plus",
                repo_type="dataset",
                filename="assets.zip",
                local_dir=assets_root,
                token=False,
            )
        )
    )
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    run_quiet(
        [
            "unzip",
            "-q",
            str(archive_path),
            "-d",
            str(extract_dir),
        ]
    )

    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    shutil.rmtree(assets_root, ignore_errors=True)

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datet

LIBERO-plus ready.


## 12. 追加学習前後を評価する

追加学習前後の2モデルを、同じ10タスク・同じseedで評価します。
評価は1モデルにつき30 rollout、2モデル合計で60 rolloutです。

In [12]:
import json
import re
from collections import deque

EVAL_CAMERA_MAPPING = {
    "agentview_image": "front",
    "robot0_eye_in_hand_image": "wrist",
}


def build_eval_command(
    policy_path: Path,
    output_dir: Path,
) -> list[str]:
    return [
        "lerobot-eval",
        f"--policy.path={policy_path}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        "--env.task=libero_spatial",
        (
            "--env.task_ids="
            + json.dumps(
                EVAL_TASK_IDS,
                separators=(",", ":"),
            )
        ),
        (
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING,
                separators=(",", ":"),
            )
        ),
        "--env.observation_height=256",
        "--env.observation_width=256",
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",
        "--eval.batch_size=1",
        (
            "--eval.n_episodes="
            f"{EVAL_EPISODES_PER_TASK}"
        ),
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={output_dir}",
    ]


def run_evaluation(
    policy_path: Path,
    output_dir: Path,
    label: str,
) -> dict:
    shutil.rmtree(
        output_dir,
        ignore_errors=True,
    )

    eval_env = os.environ.copy()
    eval_env["MUJOCO_GL"] = "egl"
    eval_env["PYTHONPATH"] = (
        str(LIBERO_PLUS_DIR)
        + os.pathsep
        + str(LEROBOT_SRC)
        + os.pathsep
        + eval_env.get("PYTHONPATH", "")
    )
    eval_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_HUB_VERBOSITY"] = "error"
    eval_env["TQDM_DISABLE"] = "1"
    eval_env["PYTHONWARNINGS"] = "ignore"
    eval_env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        build_eval_command(
            policy_path,
            output_dir,
        ),
        cwd=LEROBOT_DIR,
        env=eval_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    recent_lines: deque[str] = deque(
        maxlen=120
    )

    progress_pattern = re.compile(
        r"^EVAL_PROGRESS "
        r"task=(\d+)/(\d+) "
        r"episode=(\d+)/(\d+)$"
    )

    assert process.stdout is not None

    for raw_line in process.stdout:
        line = (
            raw_line
            .replace("\r", "")
            .strip()
        )

        if not line:
            continue

        recent_lines.append(line)
        match = progress_pattern.match(line)

        if match:
            (
                task_index,
                task_total,
                episode_index,
                episode_total,
            ) = match.groups()

            print(
                f"{label:<13} | "
                f"task {task_index}/{task_total} | "
                f"episode {episode_index}/{episode_total}"
            )

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "\n".join(recent_lines)
        )

    result_path = (
        output_dir
        / "eval_info.json"
    )

    if not result_path.is_file():
        raise FileNotFoundError(
            result_path
        )

    return json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


BASE_EVAL_INFO = run_evaluation(
    BASELINE_MODEL_DIR,
    BASE_EVAL_DIR,
    "Base model",
)

FINETUNED_EVAL_INFO = run_evaluation(
    MERGED_MODEL_DIR,
    FINETUNED_EVAL_DIR,
    "Spatial LoRA",
)

print("Evaluation complete.")

Base model    | task 1/10 | episode 1/3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Base model    | task 1/10 | episode 2/3
Base model    | task 1/10 | episode 3/3
Base model    | task 2/10 | episode 1/3
Base model    | task 2/10 | episode 2/3
Base model    | task 2/10 | episode 3/3
Base model    | task 3/10 | episode 1/3
Base model    | task 3/10 | episode 2/3
Base model    | task 3/10 | episode 3/3
Base model    | task 4/10 | episode 1/3
Base model    | task 4/10 | episode 2/3
Base model    | task 4/10 | episode 3/3
Base model    | task 5/10 | episode 1/3
Base model    | task 5/10 | episode 2/3
Base model    | task 5/10 | episode 3/3
Base model    | task 6/10 | episode 1/3
Base model    | task 6/10 | episode 2/3
Base model    | task 6/10 | episode 3/3
Base model    | task 7/10 | episode 1/3
Base model    | task 7/10 | episode 2/3
Base model    | task 7/10 | episode 3/3
Base model    | task 8/10 | episode 1/3
Base model    | task 8/10 | episode 2/3
Base model    | task 8/10 | episode 3/3
Base model    | task 9/10 | episode 1/3
Base model    | task 9/10 | episode 2/3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Spatial LoRA  | task 1/10 | episode 2/3
Spatial LoRA  | task 1/10 | episode 3/3
Spatial LoRA  | task 2/10 | episode 1/3
Spatial LoRA  | task 2/10 | episode 2/3
Spatial LoRA  | task 2/10 | episode 3/3
Spatial LoRA  | task 3/10 | episode 1/3
Spatial LoRA  | task 3/10 | episode 2/3
Spatial LoRA  | task 3/10 | episode 3/3
Spatial LoRA  | task 4/10 | episode 1/3
Spatial LoRA  | task 4/10 | episode 2/3
Spatial LoRA  | task 4/10 | episode 3/3
Spatial LoRA  | task 5/10 | episode 1/3
Spatial LoRA  | task 5/10 | episode 2/3
Spatial LoRA  | task 5/10 | episode 3/3
Spatial LoRA  | task 6/10 | episode 1/3
Spatial LoRA  | task 6/10 | episode 2/3
Spatial LoRA  | task 6/10 | episode 3/3
Spatial LoRA  | task 7/10 | episode 1/3
Spatial LoRA  | task 7/10 | episode 2/3
Spatial LoRA  | task 7/10 | episode 3/3
Spatial LoRA  | task 8/10 | episode 1/3
Spatial LoRA  | task 8/10 | episode 2/3
Spatial LoRA  | task 8/10 | episode 3/3
Spatial LoRA  | task 9/10 | episode 1/3
Spatial LoRA  | task 9/10 | episode 2/3


## 13. 成功率を比較する

`Δ (pp)`は、追加学習後から追加学習前を引いた成功率差です。

In [13]:
import pandas as pd
from IPython.display import display


def per_task_success(
    eval_info: dict,
) -> dict[int, float]:
    result: dict[int, float] = {}

    for task_info in eval_info["per_task"]:
        task_id = int(task_info["task_id"])
        successes = task_info["metrics"]["successes"]
        result[task_id] = (
            100.0
            * sum(bool(value) for value in successes)
            / len(successes)
        )

    return result


base_per_task = per_task_success(BASE_EVAL_INFO)
finetuned_per_task = per_task_success(
    FINETUNED_EVAL_INFO
)

rows = []

for task_id in EVAL_TASK_IDS:
    base_score = base_per_task[task_id]
    finetuned_score = finetuned_per_task[task_id]

    rows.append(
        {
            "Task ID": task_id,
            "Task": SPATIAL_TASK_NAMES[task_id],
            "Base (%)": base_score,
            "Spatial LoRA (%)": finetuned_score,
            "Δ (pp)": finetuned_score - base_score,
        }
    )

base_overall = float(
    BASE_EVAL_INFO["overall"]["pc_success"]
)
finetuned_overall = float(
    FINETUNED_EVAL_INFO["overall"]["pc_success"]
)

rows.append(
    {
        "Task ID": "Overall",
        "Task": "LIBERO-Spatial",
        "Base (%)": base_overall,
        "Spatial LoRA (%)": finetuned_overall,
        "Δ (pp)": finetuned_overall - base_overall,
    }
)

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
)

display(comparison_df.round(1))

print(
    f"Overall: {base_overall:.1f}% → "
    f"{finetuned_overall:.1f}% "
    f"({finetuned_overall - base_overall:+.1f} pp)"
)

,Task ID,Task,Base (%),Spatial LoRA (%),Δ (pp)
0,0,pick up the black bowl from table center and p...,100.0,100.0,0.0
1,1,pick up the black bowl next to the cookie box ...,66.7,100.0,33.3
2,2,pick up the black bowl next to the plate and p...,33.3,33.3,0.0
3,3,pick up the black bowl next to the ramekin and...,33.3,66.7,33.3
4,4,pick up the black bowl on the cookie box and p...,33.3,66.7,33.3
5,5,pick up the black bowl on the ramekin and plac...,33.3,33.3,0.0
6,6,pick up the black bowl on the stove and place ...,100.0,66.7,-33.3
7,7,pick up the black bowl on the wooden cabinet a...,100.0,66.7,-33.3
8,8,pick up the black bowl in the top drawer of th...,100.0,100.0,0.0
9,9,pick up the black bowl between the plate and t...,100.0,100.0,0.0


Overall: 70.0% → 73.3% (+3.3 pp)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 14. 学習済みモデルと比較結果をダウンロードする

In [14]:
from zipfile import ZIP_STORED, ZipFile
from google.colab import files

if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

with ZipFile(
    MERGED_ZIP_PATH,
    mode="w",
    compression=ZIP_STORED,
    allowZip64=True,
) as archive:
    for file_path in sorted(
        MERGED_MODEL_DIR.rglob("*")
    ):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=(
                    Path(MERGED_MODEL_DIR.name)
                    / file_path.relative_to(
                        MERGED_MODEL_DIR
                    )
                ),
            )

print(f"Saved: {MERGED_ZIP_PATH}")
files.download(str(MERGED_ZIP_PATH))
files.download(str(COMPARISON_CSV_PATH))

Saved: /content/smolvla_libero_plus_spatial_lora_merged.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
# ============================================
# セル1：Google Drive のマウント確認
# ============================================
from google.colab import drive
import os

# マウント済みか確認
if not os.path.exists('/content/drive/MyDrive'):
    print("🔧 Google Drive をマウント中...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive はマウント済み")

# PARC2026フォルダ準備
from pathlib import Path
PARC_DRIVE_ROOT = Path("/content/drive/MyDrive/PARC2026")
PARC_DRIVE_ROOT.mkdir(exist_ok=True)

print(f"📁 保存先: {PARC_DRIVE_ROOT}")
print(f"   存在: {PARC_DRIVE_ROOT.exists()}")

🔧 Google Drive をマウント中...
Mounted at /content/drive
📁 保存先: /content/drive/MyDrive/PARC2026
   存在: True


In [65]:
# ============================================
# セル2：学習成果物を Google Drive へ完全バックアップ
# ============================================
import shutil
import hashlib
import json
import os
import subprocess
from pathlib import Path
from datetime import datetime

# ---------- 設定 ----------
PARC_DRIVE_ROOT = Path("/content/drive/MyDrive/PARC2026")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# バージョン名（好きに変えてOK）
VERSION_NAME = f"v006_{timestamp}"  # v006_20260805_003045 のような形式

BACKUP_DIR = PARC_DRIVE_ROOT / VERSION_NAME
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"🎯 バックアップ先: {BACKUP_DIR}")
print(f"   バージョン: {VERSION_NAME}\n")

# ---------- SHA256 計算関数 ----------
def compute_sha256(file_path: Path) -> str:
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# ---------- コピー関数 ----------
def copy_with_progress(src: Path, dst: Path, label: str):
    if not src.exists():
        print(f"⚠️ {label}: 見つかりません ({src})")
        return None

    if src.is_file():
        size_mb = src.stat().st_size / 1024 / 1024
        print(f"📦 {label}: {size_mb:.1f} MB コピー中...")
        shutil.copy2(src, dst)
        sha = compute_sha256(dst)
        print(f"   ✅ 完了")
        print(f"   SHA256: {sha[:16]}...")
        return {"size_mb": size_mb, "sha256": sha}

    elif src.is_dir():
        total_size = sum(f.stat().st_size for f in src.rglob("*") if f.is_file()) / 1024 / 1024
        n_files = sum(1 for f in src.rglob("*") if f.is_file())
        print(f"📁 {label}: {total_size:.1f} MB ({n_files}ファイル) コピー中...")
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"   ✅ 完了")
        return {"size_mb": total_size, "n_files": n_files}

    return None

# ---------- メタデータ初期化 ----------
metadata = {
    "version": VERSION_NAME,
    "timestamp": timestamp,
    "backup_date": datetime.now().isoformat(),
    "files": {},
}

# ============================================
# ★★★ 必須項目
# ============================================
print("=" * 60)
print("★★★ 必須項目のバックアップ")
print("=" * 60)

# 1) merged.zip（提出物の核）
merged_zip = Path("/content/smolvla_libero_plus_spatial_lora_merged.zip")
result = copy_with_progress(
    merged_zip,
    BACKUP_DIR / "smolvla_libero_plus_spatial_lora_merged.zip",
    "merged.zip"
)
if result:
    metadata["files"]["merged_zip"] = result

print()

# 2) merged/ フォルダ（zipの中身、念のため両方保存）
merged_dir = Path("/content/smolvla_libero_plus_spatial_lora_merged")
result = copy_with_progress(
    merged_dir,
    BACKUP_DIR / "smolvla_libero_plus_spatial_lora_merged",
    "merged/ (フォルダ)"
)
if result:
    metadata["files"]["merged_dir"] = result

print()

# 3) comparison.csv（レポート用スコア）
csv_file = Path("/content/libero_spatial_comparison.csv")
result = copy_with_progress(
    csv_file,
    BACKUP_DIR / "libero_spatial_comparison.csv",
    "comparison.csv"
)
if result:
    metadata["files"]["comparison_csv"] = result
    # CSV の内容もメタデータに保存
    try:
        with open(csv_file, "r", encoding="utf-8") as f:
            metadata["comparison_csv_content"] = f.read()
    except Exception as e:
        print(f"   CSV読み込み失敗: {e}")

print()

# ============================================
# ★★ 推奨項目
# ============================================
print("=" * 60)
print("★★ 推奨項目のバックアップ")
print("=" * 60)

# 4) outputs/train/ (LoRAアダプタ・学習ログ)
outputs_dir = Path("/content/outputs")
if outputs_dir.exists():
    result = copy_with_progress(
        outputs_dir,
        BACKUP_DIR / "outputs",
        "outputs/ (LoRA・ログ)"
    )
    if result:
        metadata["files"]["outputs"] = result
else:
    print("⚠️ outputs/ が見つかりません")

print()

# 5) 環境情報
print("📋 環境情報を収集中...")
env_info = {
    "python_version": subprocess.run(
        ["python", "--version"], capture_output=True, text=True
    ).stdout.strip(),
    "colab_gpu": subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip() if os.path.exists("/usr/bin/nvidia-smi") else "N/A",
    "pip_freeze": subprocess.run(
        ["pip", "freeze"], capture_output=True, text=True
    ).stdout,
}
env_file = BACKUP_DIR / "environment.json"
env_file.write_text(json.dumps(env_info, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"   ✅ 保存: environment.json")

# 6) Notebook 本体（現在の Colab セッションのNotebookは自動保存されないので、
#    Google Drive の別の場所にNotebookが保存されているはず。
#    ここでは実行済みセルの記録として、README を作成する）
readme_content = f"""# PARC2026 学習成果物バックアップ\n\n## バージョン\n{VERSION_NAME}\n\n## 保存日時\n{datetime.now().isoformat()}\n\n## 保存内容\n- **smolvla_libero_plus_spatial_lora_merged.zip**: 提出用マージ済みモデル\n- **smolvla_libero_plus_spatial_lora_merged/**: 上記のフォルダ版\n- **libero_spatial_comparison.csv**: 学習前後のスコア比較\n- **outputs/**: 学習中間ファイル・ログ・LoRAアダプタ\n- **environment.json**: 環境情報（Python版数、GPU、pip freeze）\n\n## 復元方法\nColab で以下を実行:\n\n```python\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nimport shutil\nfrom pathlib import Path\n\nBACKUP_DIR = Path(\"/content/drive/MyDrive/PARC2026/{VERSION_NAME}\")\n\n# merged.zipを復元\nshutil.copy2(\n    BACKUP_DIR / \"smolvla_libero_plus_spatial_lora_merged.zip\",\n    \"/content/smolvla_libero_plus_spatial_lora_merged.zip\"\n)\n\n# フォルダを復元\nif not Path(\"/content/smolvla_libero_plus_spatial_lora_merged\").exists():\n    shutil.copytree(\n        BACKUP_DIR / \"smolvla_libero_plus_spatial_lora_merged\",\n        \"/content/smolvla_libero_plus_spatial_lora_merged\"\n    )\n```"""

readme_file = BACKUP_DIR / "README.md"
readme_file.write_text(readme_content, encoding="utf-8")
print(f"   ✅ 保存: README.md")

# 7) メタデータ保存
metadata_file = BACKUP_DIR / "metadata.json"
metadata_file.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"   ✅ 保存: metadata.json")

print(f"\n🎉 バックアップ完了: {BACKUP_DIR}")

🎯 バックアップ先: /content/drive/MyDrive/PARC2026/v006_20260804_141647
   バージョン: v006_20260804_141647

★★★ 必須項目のバックアップ
📦 merged.zip: 864.7 MB コピー中...
   ✅ 完了
   SHA256: 3c62f02a7777b9a9...

📁 merged/ (フォルダ): 864.7 MB (6ファイル) コピー中...
   ✅ 完了

📦 comparison.csv: 0.0 MB コピー中...
   ✅ 完了
   SHA256: fd978f4d5aa4157d...

★★ 推奨項目のバックアップ
📁 outputs/ (LoRA・ログ): 8.6 MB (14ファイル) コピー中...
   ✅ 完了

📋 環境情報を収集中...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


   ✅ 保存: environment.json
   ✅ 保存: README.md
   ✅ 保存: metadata.json

🎉 バックアップ完了: /content/drive/MyDrive/PARC2026/v006_20260804_141647


In [ ]:
# ============================================
# 復旧セル：Drive から学習成果物を復元
# ============================================
from google.colab import drive
import shutil
from pathlib import Path

# 1) Drive マウント
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive')

# 2) 復元したいバージョンを指定
# 最新のバージョンを自動選択、または手動で指定
PARC_DRIVE_ROOT = Path("/content/drive/MyDrive/PARC2026")

# 利用可能なバージョン一覧
available_versions = sorted([
    d.name for d in PARC_DRIVE_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("v")
])

print("📋 利用可能なバージョン:")
for v in available_versions:
    print(f"   - {v}")

# 最新を自動選択（手動で変更可）
VERSION_TO_RESTORE = available_versions[-1] if available_versions else None

if VERSION_TO_RESTORE is None:
    print("⚠️ バックアップが見つかりません")
else:
    print(f"\n🎯 復元するバージョン: {VERSION_TO_RESTORE}")

    BACKUP_DIR = PARC_DRIVE_ROOT / VERSION_TO_RESTORE

    # 3) merged.zip を復元
    print("\n📦 merged.zip 復元中...")
    src = BACKUP_DIR / "smolvla_libero_plus_spatial_lora_merged.zip"
    dst = Path("/content/smolvla_libero_plus_spatial_lora_merged.zip")
    if src.exists():
        shutil.copy2(src, dst)
        print(f"   ✅ 復元完了: {dst}")
    else:
        print(f"   ⚠️ ソースなし: {src}")

    # 4) merged フォルダを復元
    print("\n📁 merged/ フォルダ 復元中...")
    src_dir = BACKUP_DIR / "smolvla_libero_plus_spatial_lora_merged"
    dst_dir = Path("/content/smolvla_libero_plus_spatial_lora_merged")
    if src_dir.exists():
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_dir, dst_dir)
        print(f"   ✅ 復元完了: {dst_dir}")
    else:
        print(f"   ⚠️ ソースなし: {src_dir}")

    # 5) comparison.csv を復元
    print("\n📄 comparison.csv 復元中...")
    src = BACKUP_DIR / "libero_spatial_comparison.csv"
    dst = Path("/content/libero_spatial_comparison.csv")
    if src.exists():
        shutil.copy2(src, dst)
        print(f"   ✅ 復元完了: {dst}")

    # 6) 中身確認
    print("\n" + "=" * 60)
    print("📋 復元後の /content/ 状況")
    print("=" * 60)
    for target in [
        Path("/content/smolvla_libero_plus_spatial_lora_merged.zip"),
        Path("/content/smolvla_libero_plus_spatial_lora_merged"),
        Path("/content/libero_spatial_comparison.csv"),
    ]:
        if target.exists():
            if target.is_file():
                size = target.stat().st_size / 1024 / 1024
                print(f"✅ {target.name}: {size:.1f} MB")
            else:
                n = sum(1 for _ in target.rglob("*") if _.is_file())
                print(f"✅ {target.name}/: {n}ファイル")
        else:
            print(f"❌ {target.name}: 存在しません")

    print("\n🎉 復元完了")
    print("🎯 次のアクション:")
    print("   1. submission 作成コードを実行")
    print("   2. または validate_submission.py で検証")

In [60]:
# ============================================
# 学習後のsubmission作成（シンプル正攻法）
# ============================================
import shutil
import os
import hashlib
from pathlib import Path

# 入力
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")

# 出力
SUBMISSION_DIR = Path("/content/submission")
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 1) model_weights ディレクトリ作成 & コピー
weights_dir = SUBMISSION_DIR / "model_weights"
weights_dir.mkdir()

for f in MERGED_MODEL_DIR.rglob("*"):
    if f.is_file():
        target = weights_dir / f.relative_to(MERGED_MODEL_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)

size = sum(f.stat().st_size for f in weights_dir.rglob("*") if f.is_file())
print(f"✅ model_weights: {size / 1024 / 1024:.1f} MB")

# 2) policy_server.py 書き出し（配布テンプレの MyPolicy 部分を SmolVLA 対応に）
policy_server_code = '''#!/usr/bin/env python
"""
PARC2026 提出用 policy_server.py
SmolVLA (merged) を推論する MyPolicy 実装版
"""
import argparse
import os
import sys
from abc import ABC, abstractmethod
from pathlib import Path

import msgpack
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Request, Response


# ==========================================
# MyPolicy 実装（ここが参加者の課題）
# ==========================================

class BasePolicy(ABC):
    @abstractmethod
    def get_action(self, obs):
        pass
    @abstractmethod
    def reset(self, instruction=""):
        pass


class MyPolicy(BasePolicy):
    def __init__(self):
        # 採点環境ではlerobotがpip installされている想定
        # 学習環境と同じv0.6.0がインストールされることを期待
        from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
        from lerobot.configs import PreTrainedConfig

        weights_dir = Path(__file__).parent / "model_weights"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        config = PreTrainedConfig.from_pretrained(weights_dir)
        config.device = self.device
        if hasattr(config, "use_peft"):
            config.use_peft = False
        if hasattr(config, "load_vlm_weights"):
            config.load_vlm_weights = True

        self.policy = SmolVLAPolicy.from_pretrained(
            weights_dir, config=config, strict=False
        )
        self.policy.to(self.device)
        self.policy.eval()
        self.instruction = ""

    def get_action(self, obs):
        with torch.no_grad():
            action_tensor = self.policy.select_action(obs)
            action = action_tensor.cpu().numpy().astype(np.float32).flatten()[:7]
        return action

    def reset(self, instruction=""):
        self.instruction = instruction
        if hasattr(self.policy, "reset"):
            self.policy.reset()


# ==========================================
# 以下、サーバー部分（配布テンプレそのまま）
# 編集しないこと
# ==========================================

def deserialize_obs(data: bytes) -> dict:
    unpacked = msgpack.unpackb(data, raw=False)
    obs = {}
    for key, val in unpacked.items():
        arr = np.frombuffer(val["data"], dtype=np.dtype(val["dtype"]))
        obs[key] = arr.reshape(val["shape"]).copy()
    return obs

def serialize_action(action: np.ndarray) -> bytes:
    return msgpack.packb(
        {"data": action.astype(np.float32).tobytes()},
        use_bin_type=True,
    )

app = FastAPI(title="VLA Policy Server")
_policy = None

def set_policy(policy):
    global _policy
    _policy = policy

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reset")
async def reset_policy(request: Request):
    body = await request.body()
    instruction = ""
    seed = 0
    if body:
        import json
        data = json.loads(body)
        instruction = data.get("instruction", "")
        seed = data.get("seed", 0)
    _policy.reset(instruction=instruction)
    return {"status": "ok"}

@app.post("/act")
async def act(request: Request):
    body = await request.body()
    obs = deserialize_obs(body)
    action = _policy.get_action(obs)
    return Response(
        content=serialize_action(action),
        media_type="application/x-msgpack",
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--host", type=str, default="0.0.0.0")
    args = parser.parse_args()
    set_policy(MyPolicy())
    uvicorn.run(app, host=args.host, port=args.port)
'''

(SUBMISSION_DIR / "policy_server.py").write_text(policy_server_code, encoding="utf-8")
print("✅ policy_server.py 書き出し")

# 3) requirements.txt（k-aiki253等の成功者の書式を参考）
requirements = """# Policy server dependencies
fastapi>=0.68
uvicorn>=0.15
msgpack>=1.0
numpy>=1.19

# lerobot v0.6.0互換のライブラリ
# 採点環境の Python 3.10 で lerobot をインストールできる形にする
# ※ lerobot本体は同梱しないので、pip install に依存する
# ※ もし採点環境で lerobot が入らなければ、別戦略に切り替え

# lerobot の間接依存を明示
transformers>=4.30
safetensors
tokenizers
huggingface_hub
peft
draccus
num2words
einops
typing_extensions
"""

(SUBMISSION_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print("✅ requirements.txt 書き出し")

# 4) zip作成
SUBMISSION_ZIP = "/content/submission_simple.zip"
if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

# 重要：フォルダを介さず、中身を直接 zip 化
shutil.make_archive("/content/submission_simple", "zip", SUBMISSION_DIR)

size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\n📦 zip サイズ: {size_mb:.1f} MB")

# 5) SHA256
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

# 6) 中身確認
import zipfile
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    print("\n📋 zip中身:")
    for name in sorted(z.namelist())[:15]:
        print(f"   {name}")
    print(f"   ... 全 {len(z.namelist())} ファイル")

print("\n🎯 次: /content/submission_simple.zip をダウンロードして...")
print("   選択肢A: Omnicampusにそのまま提出")
print("   選択肢B: Mac上のDockerで validate/evaluate.py 実行")

✅ model_weights: 864.7 MB
✅ policy_server.py 書き出し
✅ requirements.txt 書き出し

📦 zip サイズ: 685.4 MB
🔐 SHA256: cd51c5e2d5f21afa531bb5a526cc7baeb9ec233fd9bc3ac692a20f1a79f3658c

📋 zip中身:
   model_weights/
   model_weights/config.json
   model_weights/model.safetensors
   model_weights/policy_postprocessor.json
   model_weights/policy_postprocessor_step_0_unnormalizer_processor.safetensors
   model_weights/policy_preprocessor.json
   model_weights/policy_preprocessor_step_5_normalizer_processor.safetensors
   policy_server.py
   requirements.txt
   ... 全 9 ファイル

🎯 次: /content/submission_simple.zip をダウンロードして...
   選択肢A: Omnicampusにそのまま提出
   選択肢B: Mac上のDockerで validate/evaluate.py 実行


In [61]:
# ============================================
# Colab 上の Python 3.10 venv で近似 validate
# ============================================

print("🔧 Python 3.10 venv セットアップ...")
!apt-get install -y python3.10 python3.10-venv python3.10-dev 2>&1 | tail -1
!rm -rf /tmp/py310_v
!python3.10 -m venv /tmp/py310_v
!/tmp/py310_v/bin/pip install --upgrade pip -q
!/tmp/py310_v/bin/pip install -q msgpack numpy requests fastapi uvicorn packaging psutil

print("\n🔧 validate_submission.py 取得...")
!wget -q https://raw.githubusercontent.com/matsuolab/PARC2026_pre/main/validate_submission.py \
    -O /tmp/validate_submission.py

print("\n" + "="*60)
print("📋 静的チェック（--static）")
print("="*60)
!/tmp/py310_v/bin/python /tmp/validate_submission.py /content/submission_simple.zip --static

print("\n" + "="*60)
print("🚀 完全検証（--install）")
print("="*60)
print("⚠️ lerobot のインストールに時間がかかる可能性あり")
!/tmp/py310_v/bin/python /tmp/validate_submission.py /content/submission_simple.zip --install

🔧 Python 3.10 venv セットアップ...
0 upgraded, 0 newly installed, 0 to remove and 164 not upgraded.

🔧 validate_submission.py 取得...

📋 静的チェック（--static）
バリデーション結果: PASS  (errors=0, warnings=0)

🚀 完全検証（--install）
⚠️ lerobot のインストールに時間がかかる可能性あり
バリデーション結果: FAIL  (errors=1, warnings=0)
  [·] INFO  smoke.installing: requirements.txt をインストール中...
  [✗] ERROR smoke.server_died: policy_server.py が起動直後に終了しました (exit=1)。
      Traceback (most recent call last):
        File "/tmp/valdyn_zv9ymqvl/policy_server.py", line 126, in <module>
          set_policy(MyPolicy())
        File "/tmp/valdyn_zv9ymqvl/policy_server.py", line 36, in __init__
          from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
      ModuleNotFoundError: No module named 'lerobot'


In [52]:
# ============================================
# セル1：既存資産の確認
# ============================================
from pathlib import Path
import os

# 1) 学習済みモデルの存在確認
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")
print(f"📦 学習済みモデル存在: {MERGED_MODEL_DIR.exists()}")

if MERGED_MODEL_DIR.exists():
    # 中身の確認
    for f in sorted(MERGED_MODEL_DIR.iterdir()):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"   {f.name}: {size_mb:.1f} MB")

# 2) 学習時のlerobotが残っているか確認
LEROBOT_DIR = Path("/content/lerobot/src/lerobot")
print(f"\n📦 lerobot存在: {LEROBOT_DIR.exists()}")

# 3) もし学習済みモデルが消えていたら Google Drive から復元
if not MERGED_MODEL_DIR.exists():
    print("\n⚠️ モデルが見つかりません")
    print("→ Google Drive に保存済みの merged.zip をアップロードしてください")

📦 学習済みモデル存在: True
   config.json: 0.0 MB
   model.safetensors: 864.7 MB
   policy_postprocessor.json: 0.0 MB
   policy_postprocessor_step_0_unnormalizer_processor.safetensors: 0.0 MB
   policy_preprocessor.json: 0.0 MB
   policy_preprocessor_step_5_normalizer_processor.safetensors: 0.0 MB

📦 lerobot存在: True


In [53]:
# ============================================
# 修正版セル2〜4（統合版）：全コピー + typing.Selfパッチ
# ============================================
import shutil
import re
from pathlib import Path

# 元のlerobot
SRC_LEROBOT = Path("/content/lerobot/src/lerobot")

# 提出フォルダをクリーンに準備
SUBMISSION_DIR = Path("/content/submission")
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 抽出先
DST_LEROBOT = SUBMISSION_DIR / "lerobot"

# ⭐ 変更点：lerobot全部をコピー（ただし__pycache__は除外）
def ignore_patterns(dir, files):
    return [f for f in files if f == "__pycache__" or f.endswith(".pyc")]

shutil.copytree(SRC_LEROBOT, DST_LEROBOT, ignore=ignore_patterns)

# サイズ確認
total_size = sum(f.stat().st_size for f in DST_LEROBOT.rglob("*") if f.is_file())
print(f"📊 lerobot 全コピー後サイズ: {total_size / 1024 / 1024:.1f} MB")

# ⭐ typing.Self を typing_extensions.Self にパッチ
def patch_typing_self(directory):
    """
    ディレクトリ内の全.pyファイルで typing.Self を typing_extensions.Self に置換
    """
    patched_files = []

    for py_file in Path(directory).rglob("*.py"):
        content = py_file.read_text(encoding="utf-8")
        original = content

        # パターン: from typing import ..., Self, ...
        pattern = r"from typing import ([^;\n]*)"

        def replace_typing_self(m):
            imports = m.group(1)
            # Selfが含まれていなければ変更しない
            if "Self" not in imports.split():
                # ", Self" や "Self," も検出
                if not re.search(r"\bSelf\b", imports):
                    return m.group(0)

            # Selfを除去
            parts = [p.strip() for p in imports.split(",")]
            parts_without_self = [p for p in parts if p != "Self" and p.strip() != "Self"]

            if parts_without_self:
                new_typing_import = f"from typing import {', '.join(parts_without_self)}"
                return f"{new_typing_import}\nfrom typing_extensions import Self"
            else:
                return "from typing_extensions import Self"

        content = re.sub(pattern, replace_typing_self, content)

        if content != original:
            py_file.write_text(content, encoding="utf-8")
            patched_files.append(py_file.name)

    return patched_files

patched_files = patch_typing_self(DST_LEROBOT)
print(f"\n✅ typing.Self をパッチしたファイル: {len(patched_files)}件")

# 確認：まだ残っているか？
import subprocess
result = subprocess.run(
    ["grep", "-R", "-l", "from typing import.*Self", str(DST_LEROBOT)],
    capture_output=True, text=True,
)
remaining = result.stdout.strip().splitlines() if result.stdout.strip() else []
if remaining:
    print(f"\n⚠️ まだ残っているファイル ({len(remaining)}件):")
    for f in remaining[:5]:
        print(f"   {f}")
else:
    print("\n✅ typing.Self の直接import はすべて修正済み")

# ⭐ __version__.py の存在確認
version_file = DST_LEROBOT / "__version__.py"
if version_file.exists():
    print(f"\n✅ __version__.py 存在: {version_file}")
    print(f"   内容: {version_file.read_text()[:200]}")
else:
    print(f"\n⚠️ __version__.py が存在しません")
    # __version__.py を自作
    version_file.write_text('__version__ = "0.6.0"\n', encoding="utf-8")
    print(f"   → 自作しました")

📊 lerobot 全コピー後サイズ: 5.7 MB

✅ typing.Self をパッチしたファイル: 1件

✅ typing.Self の直接import はすべて修正済み

✅ __version__.py 存在: /content/submission/lerobot/__version__.py
   内容: #!/usr/bin/env python

# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compli


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [54]:
# ============================================
# セル5：学習済みモデルをsubmissionへコピー
# ============================================

MODEL_WEIGHTS_DIR = SUBMISSION_DIR / "model_weights"
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# merged.zipの中身をコピー
for file_path in MERGED_MODEL_DIR.rglob("*"):
    if file_path.is_file():
        target = MODEL_WEIGHTS_DIR / file_path.relative_to(MERGED_MODEL_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, target)

# サイズ確認
weights_size = sum(f.stat().st_size for f in MODEL_WEIGHTS_DIR.rglob("*") if f.is_file())
print(f"✅ model_weights: {weights_size / 1024 / 1024:.1f} MB")

✅ model_weights: 864.7 MB


In [55]:
# ============================================
# セル6：policy_server.py を書き出し
# ============================================

policy_server_code = r'''"""
ポリシーサーバー v006 - 案A: 最小lerobot同梱版
"""
import argparse
import os
import sys
from abc import ABC, abstractmethod
from pathlib import Path

# 同梱lerobotをimportパスに追加
sys.path.insert(0, str(Path(__file__).parent))

import lerobot
import msgpack
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Request, Response

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs import PreTrainedConfig


class BasePolicy(ABC):
    @abstractmethod
    def get_action(self, obs):
        pass
    @abstractmethod
    def reset(self, instruction=""):
        pass


class MyPolicy(BasePolicy):
    def __init__(self):
        weights_dir = Path(__file__).parent / "model_weights"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        config = PreTrainedConfig.from_pretrained(weights_dir)
        config.device = self.device
        config.use_peft = False
        config.load_vlm_weights = True

        self.policy = SmolVLAPolicy.from_pretrained(
            weights_dir, config=config, strict=False
        )
        self.policy.to(self.device)
        self.policy.eval()
        self.instruction = ""

    def get_action(self, obs):
        with torch.no_grad():
            action_tensor = self.policy.select_action(obs)
            action = action_tensor.cpu().numpy().astype(np.float32).flatten()[:7]
        return action

    def reset(self, instruction=""):
        self.instruction = instruction
        if hasattr(self.policy, "reset"):
            self.policy.reset()


# サーバー部分（変更不可）
def deserialize_obs(data: bytes) -> dict:
    unpacked = msgpack.unpackb(data, raw=False)
    obs = {}
    for key, val in unpacked.items():
        arr = np.frombuffer(val["data"], dtype=np.dtype(val["dtype"]))
        obs[key] = arr.reshape(val["shape"]).copy()
    return obs

def serialize_action(action: np.ndarray) -> bytes:
    return msgpack.packb(
        {"data": action.astype(np.float32).tobytes()},
        use_bin_type=True,
    )

app = FastAPI(title="VLA Policy Server")
_policy = None

def set_policy(policy):
    global _policy
    _policy = policy

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reset")
async def reset_policy(request: Request):
    body = await request.body()
    instruction = ""
    if body:
        import json
        data = json.loads(body)
        instruction = data.get("instruction", "")
    _policy.reset(instruction=instruction)
    return {"status": "ok"}

@app.post("/act")
async def act(request: Request):
    body = await request.body()
    obs = deserialize_obs(body)
    action = _policy.get_action(obs)
    return Response(
        content=serialize_action(action),
        media_type="application/x-msgpack",
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--host", type=str, default="0.0.0.0")
    args = parser.parse_args()
    set_policy(MyPolicy())
    uvicorn.run(app, host=args.host, port=args.port)
'''

(SUBMISSION_DIR / "policy_server.py").write_text(policy_server_code, encoding="utf-8")
print(f"✅ policy_server.py 書き出し完了")

✅ policy_server.py 書き出し完了


In [56]:
# ============================================
# submission_v007 作成（Colabでのimport検証をスキップ）
# ============================================
import shutil
import hashlib
import os
from pathlib import Path

CHOSEN_SRC_LEROBOT = Path("/tmp/lerobot_v0.4.0/src/lerobot")
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")
SUBMISSION_DIR = Path("/content/submission")

# クリーンに準備
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 1) lerobot v0.4.0 をそのままコピー（policies/__init__.py も元に戻した状態）
DST_LEROBOT = SUBMISSION_DIR / "lerobot"

def ignore_patterns(dir, files):
    return [f for f in files if f == "__pycache__" or f.endswith(".pyc")]

shutil.copytree(CHOSEN_SRC_LEROBOT, DST_LEROBOT, ignore=ignore_patterns)

size = sum(f.stat().st_size for f in DST_LEROBOT.rglob("*") if f.is_file())
print(f"✅ lerobot v0.4.0: {size / 1024 / 1024:.1f} MB")

# 2) __version__.py 確認
version_file = DST_LEROBOT / "__version__.py"
if not version_file.exists():
    version_file.write_text('__version__ = "0.4.0"\n', encoding="utf-8")
    print("✅ __version__.py 自作")
else:
    print(f"✅ __version__.py 存在: {version_file.read_text()[:100]}")

# 3) model_weights コピー
MODEL_WEIGHTS_DIR = SUBMISSION_DIR / "model_weights"
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

for file_path in MERGED_MODEL_DIR.rglob("*"):
    if file_path.is_file():
        target = MODEL_WEIGHTS_DIR / file_path.relative_to(MERGED_MODEL_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, target)

weights_size = sum(f.stat().st_size for f in MODEL_WEIGHTS_DIR.rglob("*") if f.is_file())
print(f"✅ model_weights: {weights_size / 1024 / 1024:.1f} MB")

# 4) policy_server.py 書き出し
policy_server_code = r'''"""ポリシーサーバー v007"""
import argparse
import os
import sys
from abc import ABC, abstractmethod
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

import lerobot
import msgpack
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Request, Response

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs import PreTrainedConfig


class BasePolicy(ABC):
    @abstractmethod
    def get_action(self, obs):
        pass
    @abstractmethod
    def reset(self, instruction=""):
        pass


class MyPolicy(BasePolicy):
    def __init__(self):
        weights_dir = Path(__file__).parent / "model_weights"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        config = PreTrainedConfig.from_pretrained(weights_dir)
        config.device = self.device
        config.use_peft = False
        if hasattr(config, "load_vlm_weights"):
            config.load_vlm_weights = True

        self.policy = SmolVLAPolicy.from_pretrained(
            weights_dir, config=config, strict=False
        )
        self.policy.to(self.device)
        self.policy.eval()
        self.instruction = ""

    def get_action(self, obs):
        with torch.no_grad():
            action_tensor = self.policy.select_action(obs)
            action = action_tensor.cpu().numpy().astype(np.float32).flatten()[:7]
        return action

    def reset(self, instruction=""):
        self.instruction = instruction
        if hasattr(self.policy, "reset"):
            self.policy.reset()


def deserialize_obs(data):
    unpacked = msgpack.unpackb(data, raw=False)
    obs = {}
    for key, val in unpacked.items():
        arr = np.frombuffer(val["data"], dtype=np.dtype(val["dtype"]))
        obs[key] = arr.reshape(val["shape"]).copy()
    return obs

def serialize_action(action):
    return msgpack.packb(
        {"data": action.astype(np.float32).tobytes()},
        use_bin_type=True,
    )

app = FastAPI(title="VLA Policy Server")
_policy = None

def set_policy(policy):
    global _policy
    _policy = policy

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reset")
async def reset_policy(request: Request):
    body = await request.body()
    instruction = ""
    if body:
        import json
        data = json.loads(body)
        instruction = data.get("instruction", "")
    _policy.reset(instruction=instruction)
    return {"status": "ok"}

@app.post("/act")
async def act(request: Request):
    body = await request.body()
    obs = deserialize_obs(body)
    action = _policy.get_action(obs)
    return Response(
        content=serialize_action(action),
        media_type="application/x-msgpack",
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--host", type=str, default="0.0.0.0")
    args = parser.parse_args()
    set_policy(MyPolicy())
    uvicorn.run(app, host=args.host, port=args.port)
'''

(SUBMISSION_DIR / "policy_server.py").write_text(policy_server_code, encoding="utf-8")
print("✅ policy_server.py 書き出し")

# 5) requirements.txt（pyserial 追加）
requirements = """# Policy server dependencies
# ※ プリインストール済みは書かない
# ※ lerobot は同梱するため書かない

# lerobot v0.4.0 の副作用依存
pyserial

# lerobot v0.4.0 が要求する固有依存
peft
draccus
num2words

# 保険（プリインストール 4.53.2 と互換ない場合）
transformers==4.57.6
"""

(SUBMISSION_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print("✅ requirements.txt 書き出し")

# 6) zip作成
SUBMISSION_ZIP = "/content/submission_v007.zip"
if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
shutil.make_archive("/content/submission_v007", "zip", SUBMISSION_DIR)

size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\n📦 zip サイズ: {size_mb:.1f} MB")

# 7) SHA256
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

print("\n🎯 次: Python 3.10 venv で validate_submission.py 実行")

✅ lerobot v0.4.0: 2.3 MB
✅ __version__.py 存在: #!/usr/bin/env python

# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed
✅ model_weights: 864.7 MB
✅ policy_server.py 書き出し
✅ requirements.txt 書き出し

📦 zip サイズ: 686.1 MB
🔐 SHA256: 2cff26bc1acb297f076bebeda1a0a92b0c059f87933bdacc54c100a87fdaeb17

🎯 次: Python 3.10 venv で validate_submission.py 実行


In [58]:
# ============================================
# groot ディレクトリを削除（保険策）
# ============================================
import shutil
from pathlib import Path

DST_LEROBOT = Path("/content/submission/lerobot")

# groot削除
groot_dir = DST_LEROBOT / "policies" / "groot"
if groot_dir.exists():
    shutil.rmtree(groot_dir)
    print("✅ groot ディレクトリを削除")

# policies/__init__.py から groot の import を削除
policies_init = DST_LEROBOT / "policies" / "__init__.py"
content = policies_init.read_text(encoding="utf-8")

# groot 関連の行を削除
new_lines = []
for line in content.splitlines():
    if "groot" not in line.lower():
        new_lines.append(line)

policies_init.write_text("\n".join(new_lines), encoding="utf-8")
print("✅ policies/__init__.py から groot import を削除")

# zip再作成
import os
SUBMISSION_ZIP = "/content/submission_v007.zip"
if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

shutil.make_archive("/content/submission_v007", "zip", Path("/content/submission"))
size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\n📦 修正版 zip サイズ: {size_mb:.1f} MB")

✅ groot ディレクトリを削除
✅ policies/__init__.py から groot import を削除

📦 修正版 zip サイズ: 686.1 MB


In [59]:
# ============================================
# Python 3.10 venv で validate_submission.py 実行
# ============================================

print("🔧 Step 1: Python 3.10 セットアップ...")
!apt-get install -y python3.10 python3.10-venv python3.10-dev 2>&1 | tail -3

print("\n🔧 Step 2: venv 作成...")
!rm -rf /tmp/py310_venv
!python3.10 -m venv /tmp/py310_venv

print("\n🔧 Step 3: 検証ツールインストール...")
!/tmp/py310_venv/bin/pip install --upgrade pip -q
!/tmp/py310_venv/bin/pip install -q msgpack numpy requests fastapi uvicorn packaging psutil

print("\n🔧 Step 4: validate_submission.py 取得...")
!wget -q https://raw.githubusercontent.com/matsuolab/PARC2026_pre/main/validate_submission.py \
    -O /tmp/validate_submission.py

print("\n" + "="*60)
print("📋 Step 5A: 静的チェック（--static、30秒）")
print("="*60)
!/tmp/py310_venv/bin/python /tmp/validate_submission.py /content/submission_v007.zip --static

print("\n" + "="*60)
print("🚀 Step 5B: 完全検証（--install、5〜15分）")
print("="*60)
print("⚠️ 依存インストールとサーバー起動に時間がかかります")
!/tmp/py310_venv/bin/python /tmp/validate_submission.py /content/submission_v007.zip --install

🔧 Step 1: Python 3.10 セットアップ...
python3.10-dev is already the newest version (3.10.12-1~22.04.16).
python3.10-venv is already the newest version (3.10.12-1~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 164 not upgraded.

🔧 Step 2: venv 作成...

🔧 Step 3: 検証ツールインストール...

🔧 Step 4: validate_submission.py 取得...

📋 Step 5A: 静的チェック（--static、30秒）
バリデーション結果: PASS  (errors=0, warnings=0)

🚀 Step 5B: 完全検証（--install、5〜15分）
⚠️ 依存インストールとサーバー起動に時間がかかります
バリデーション結果: FAIL  (errors=1, warnings=0)
  [·] INFO  smoke.installing: requirements.txt をインストール中...
  [✗] ERROR smoke.server_died: policy_server.py が起動直後に終了しました (exit=1)。
        File "/tmp/valdyn_ios4i3ge/policy_server.py", line 17, in <module>
          from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
        File "/tmp/valdyn_ios4i3ge/lerobot/policies/__init__.py", line 15, in <module>
          from .act.configuration_act import ACTConfig as ACTConfig
        File "/tmp/valdyn_ios4i3ge/lerobot/policies/act/configurati

In [38]:
# ============================================
# セル7（修正版）：requirements.txt を書き出し
# ============================================

requirements = """# Policy server dependencies
fastapi>=0.68
uvicorn>=0.15
msgpack>=1.0
numpy>=1.19

# lerobot が使う軽量ライブラリ
transformers>=4.30
safetensors
tokenizers
huggingface_hub
peft
draccus
num2words
einops
typing_extensions

# ⭐ 追加：他ポリシー（ACT/Diffusion/VQBet）が要求する依存
torchvision
opencv-python-headless
pillow
"""

(SUBMISSION_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print(f"✅ requirements.txt を書き出しました（torchvision追加）")

# 中身確認
print("\n📋 requirements.txt の内容:")
print((SUBMISSION_DIR / "requirements.txt").read_text())

✅ requirements.txt を書き出しました（torchvision追加）

📋 requirements.txt の内容:
# Policy server dependencies
fastapi>=0.68
uvicorn>=0.15
msgpack>=1.0
numpy>=1.19

# lerobot が使う軽量ライブラリ
transformers>=4.30
safetensors
tokenizers
huggingface_hub
peft
draccus
num2words
einops
typing_extensions

# ⭐ 追加：他ポリシー（ACT/Diffusion/VQBet）が要求する依存
torchvision
opencv-python-headless
pillow



In [39]:
# ============================================
# 追加セル：policies/__init__.py の軽量化
# ============================================

policies_init = SUBMISSION_DIR / "lerobot" / "policies" / "__init__.py"

# 現在の内容を確認
if policies_init.exists():
    print("📋 現在の内容（一部）:")
    print(policies_init.read_text()[:500])
    print("...")

# ⭐ SmolVLAだけをimportする最小版に書き換え
new_content = '''"""LeRobot policies (SmolVLA only for submission)"""

# ⭐ 変更前：ACT/Diffusion/VQBet等を全部import
# ⭐ 変更後：SmolVLAだけをimport

# SmolVLAだけをimport
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# 共通基底クラス
from lerobot.policies.pretrained import PreTrainedPolicy

__all__ = [
    "SmolVLAConfig",
    "SmolVLAPolicy",
    "PreTrainedPolicy",
]
'''

policies_init.write_text(new_content, encoding="utf-8")
print(f"\n✅ policies/__init__.py を軽量化しました")
print(f"   （SmolVLA以外のimportを削除）")

# 確認
print("\n📋 修正後の内容:")
print(policies_init.read_text())

📋 現在の内容（一部）:
# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or i
...

✅ policies/__init__.py を軽量化しました
   （SmolVLA以外のimportを削除）

📋 修正後の内容:
"""LeRobot policies (SmolVLA only for submission)"""

# ⭐ 変更前：ACT/Diffusion/VQBet等を全部import
# ⭐ 変更後：SmolVLAだけをimport

# SmolVLAだけをimport
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# 共通基底クラス
from lerobot.policies.pretrained import PreTrainedPolicy

__all__ = [
    "SmolVLAConfig",
    "SmolVLAPolicy",
    "PreTrai

In [40]:
# ============================================
# セル8：submission.zip 作成
# ============================================
import hashlib
from google.colab import files

SUBMISSION_ZIP = "/content/submission_v006.zip"

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

shutil.make_archive("/content/submission_v006", "zip", SUBMISSION_DIR)

# サイズ
size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"📦 zip サイズ: {size_mb:.1f} MB")

# ハッシュ
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

# 中身の一覧
import zipfile
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    print("\n📋 zip中身（上位30件）:")
    for name in sorted(z.namelist())[:30]:
        print(f"   {name}")
    print(f"\n   合計 {len(z.namelist())} ファイル")
print(f"\n✅ 作成完了: {SUBMISSION_ZIP}")

📦 zip サイズ: 687.1 MB
🔐 SHA256: 6d80caad18d98785122b377dc466e35fbbf5ac26c47d19830740a2989964fb48

📋 zip中身（上位30件）:
   lerobot/
   lerobot/__init__.py
   lerobot/__version__.py
   lerobot/annotations/
   lerobot/annotations/__init__.py
   lerobot/annotations/steerable_pipeline/
   lerobot/annotations/steerable_pipeline/__init__.py
   lerobot/annotations/steerable_pipeline/config.py
   lerobot/annotations/steerable_pipeline/executor.py
   lerobot/annotations/steerable_pipeline/frames.py
   lerobot/annotations/steerable_pipeline/modules/
   lerobot/annotations/steerable_pipeline/modules/__init__.py
   lerobot/annotations/steerable_pipeline/modules/general_vqa.py
   lerobot/annotations/steerable_pipeline/modules/interjections_and_speech.py
   lerobot/annotations/steerable_pipeline/modules/plan_subtasks_memory.py
   lerobot/annotations/steerable_pipeline/prompts/
   lerobot/annotations/steerable_pipeline/prompts/__init__.py
   lerobot/annotations/steerable_pipeline/prompts/interjections_initia

In [41]:
# ============================================
# セル8（再実行）：submission.zip 再作成
# ============================================
import hashlib
import os
import shutil

SUBMISSION_ZIP = "/content/submission_v006.zip"

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

shutil.make_archive("/content/submission_v006", "zip", SUBMISSION_DIR)

size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"📦 zip サイズ: {size_mb:.1f} MB")

sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

📦 zip サイズ: 687.1 MB
🔐 SHA256: 6d80caad18d98785122b377dc466e35fbbf5ac26c47d19830740a2989964fb48


In [42]:
# ============================================
# セル9：Python 3.10 環境で validate_submission.py 実行
# ============================================
print("🔧 Step 1: Python 3.10 セットアップ...")
!apt-get install -y python3.10 python3.10-venv python3.10-dev 2>&1 | tail -3

print("\n🔧 Step 2: venv 作成...")
!rm -rf /tmp/py310_venv
!python3.10 -m venv /tmp/py310_venv

print("\n🔧 Step 3: 検証ツールインストール...")
!/tmp/py310_venv/bin/pip install --upgrade pip -q
!/tmp/py310_venv/bin/pip install -q msgpack numpy requests fastapi uvicorn packaging psutil

print("\n🔧 Step 4: validate_submission.py 取得...")
!wget -q https://raw.githubusercontent.com/matsuolab/PARC2026_pre/main/validate_submission.py \
    -O /tmp/validate_submission.py

print("\n" + "="*60)
print("📋 Step 5A: 静的チェック（--static、30秒）")
print("="*60)
!/tmp/py310_venv/bin/python /tmp/validate_submission.py {SUBMISSION_ZIP} --static

print("\n" + "="*60)
print("🚀 Step 5B: 完全検証（--install、5〜15分）")
print("="*60)
print("⚠️ 依存インストールとサーバー起動に時間がかかります")
!/tmp/py310_venv/bin/python /tmp/validate_submission.py {SUBMISSION_ZIP} --install

🔧 Step 1: Python 3.10 セットアップ...
python3.10-dev is already the newest version (3.10.12-1~22.04.16).
python3.10-venv is already the newest version (3.10.12-1~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 164 not upgraded.

🔧 Step 2: venv 作成...

🔧 Step 3: 検証ツールインストール...

🔧 Step 4: validate_submission.py 取得...

📋 Step 5A: 静的チェック（--static、30秒）
バリデーション結果: PASS  (errors=0, warnings=0)

🚀 Step 5B: 完全検証（--install、5〜15分）
⚠️ 依存インストールとサーバー起動に時間がかかります
バリデーション結果: FAIL  (errors=1, warnings=0)
  [·] INFO  smoke.installing: requirements.txt をインストール中...
  [✗] ERROR smoke.server_died: policy_server.py が起動直後に終了しました (exit=1)。
          from .configuration_smolvla import SmolVLAConfig
        File "/tmp/valdyn_crz74oif/lerobot/policies/smolvla/configuration_smolvla.py", line 17, in <module>
          from lerobot.configs import FeatureType, NormalizationMode, PolicyFeature, PreTrainedConfig
        File "/tmp/valdyn_crz74oif/lerobot/configs/__init__.py", line 26, in <module>
          from .poli

In [43]:
# ============================================
# lerobot 古いバージョンの検証
# ============================================
import subprocess
import shutil
from pathlib import Path

# v0.5.0 と v0.4.0 を試す（両方cloneして中身確認）
TAGS_TO_CHECK = ["v0.5.0", "v0.4.0"]

for tag in TAGS_TO_CHECK:
    print(f"\n{'='*60}")
    print(f"📥 lerobot {tag} を検証")
    print(f"{'='*60}")

    tag_dir = Path(f"/tmp/lerobot_{tag}")
    if tag_dir.exists():
        shutil.rmtree(tag_dir)

    # Clone
    result = subprocess.run([
        "git", "clone", "--branch", tag, "--depth", "1",
        "https://github.com/huggingface/lerobot.git", str(tag_dir)
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"❌ Clone失敗: {result.stderr[:500]}")
        continue

    # lerobot本体パスを見つける
    if (tag_dir / "src" / "lerobot").exists():
        lerobot_src = tag_dir / "src" / "lerobot"
    elif (tag_dir / "lerobot").exists():
        lerobot_src = tag_dir / "lerobot"
    else:
        print(f"❌ lerobot本体が見つかりません")
        continue

    print(f"✅ Clone成功: {lerobot_src}")

    # SmolVLA の存在確認
    smolvla_path = lerobot_src / "policies" / "smolvla"
    if smolvla_path.exists():
        print(f"✅ SmolVLA存在: {smolvla_path}")
        print(f"   内容: {[f.name for f in smolvla_path.iterdir()][:10]}")
    else:
        print(f"❌ SmolVLA が存在しません")
        continue

    # typing.Self の使用箇所
    result_self = subprocess.run(
        ["grep", "-R", "-l", "from typing import.*Self", str(lerobot_src)],
        capture_output=True, text=True
    )
    n_self = len(result_self.stdout.strip().splitlines()) if result_self.stdout.strip() else 0
    print(f"📊 typing.Self 使用ファイル数: {n_self}")

    # PEP 695 構文の使用箇所（[T: Type] 形式）
    result_pep695 = subprocess.run(
        ["grep", "-R", "-l", "-E", r"def [a-zA-Z_]+\[.*:.*\]\(", str(lerobot_src)],
        capture_output=True, text=True
    )
    n_pep695 = len(result_pep695.stdout.strip().splitlines()) if result_pep695.stdout.strip() else 0
    print(f"📊 PEP 695構文 使用ファイル数: {n_pep695}")

    # 判定
    if n_pep695 == 0:
        print(f"\n🎉 {tag} は Python 3.10 互換の可能性大！")
        if n_self > 0:
            print(f"   （ただし typing.Self は {n_self}箇所あるので、パッチ必要）")
    else:
        print(f"\n⚠️ {tag} には PEP 695 構文があるため Python 3.10 で動きません")


📥 lerobot v0.5.0 を検証


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Clone成功: /tmp/lerobot_v0.5.0/src/lerobot
✅ SmolVLA存在: /tmp/lerobot_v0.5.0/src/lerobot/policies/smolvla
   内容: ['modeling_smolvla.py', 'configuration_smolvla.py', 'processor_smolvla.py', 'smolvlm_with_expert.py', 'README.md']
📊 typing.Self 使用ファイル数: 0
📊 PEP 695構文 使用ファイル数: 1

⚠️ v0.5.0 には PEP 695 構文があるため Python 3.10 で動きません

📥 lerobot v0.4.0 を検証


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Clone成功: /tmp/lerobot_v0.4.0/src/lerobot
✅ SmolVLA存在: /tmp/lerobot_v0.4.0/src/lerobot/policies/smolvla
   内容: ['modeling_smolvla.py', 'configuration_smolvla.py', 'processor_smolvla.py', 'smolvlm_with_expert.py', 'README.md']
📊 typing.Self 使用ファイル数: 0
📊 PEP 695構文 使用ファイル数: 0

🎉 v0.4.0 は Python 3.10 互換の可能性大！


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [44]:
# ============================================
# v0.4.0 で v0.6.0 学習済みモデルが読めるか確認
# ============================================

import sys
import gc
from pathlib import Path

# 既存の lerobot を消す
for mod in list(sys.modules):
    if mod == "lerobot" or mod.startswith("lerobot."):
        del sys.modules[mod]

gc.collect()

# v0.4.0 clone
!rm -rf /tmp/lerobot_v0.4.0
!git clone --branch v0.4.0 --depth 1 \
    https://github.com/huggingface/lerobot.git \
    /tmp/lerobot_v0.4.0

# import path
sys.path.insert(0, "/tmp/lerobot_v0.4.0/src")

print("=== import test ===")

try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from lerobot.configs import PreTrainedConfig
    print("✅ import成功")
except Exception as e:
    print("❌ import失敗")
    raise

MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")

print("\n=== config load ===")

try:
    config = PreTrainedConfig.from_pretrained(MODEL_DIR)
    print("✅ config読込成功")
    print(type(config))
except Exception as e:
    print("❌ config読込失敗")
    raise

print("\n=== model load ===")

try:
    config.device = "cpu"

    policy = SmolVLAPolicy.from_pretrained(
        MODEL_DIR,
        config=config,
        strict=False,
    )

    print("✅ model読込成功")
    print(type(policy))

except Exception as e:
    print("❌ model読込失敗")
    raise

Cloning into '/tmp/lerobot_v0.4.0'...
remote: Enumerating objects: 628, done.
remote: Counting objects: 100% (628/628), done.
remote: Compressing objects: 100% (532/532), done.
remote: Total 628 (delta 72), reused 613 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (628/628), 5.91 MiB | 23.54 MiB/s, done.
Resolving deltas: 100% (72/72), done.
Note: switching to 'f25ac02e6c8fa9c467ab8462289e5f4aed3a2e85'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Filtering content: 100% (45/45), 69.03 MiB | 4.43 MiB/s, done.
=

TypeError: non-default argument 'backbone_cfg' follows default argument

In [45]:
import traceback
try:

    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

except Exception:

    traceback.print_exc(limit=50)

Traceback (most recent call last):
  File "/tmp/ipykernel_2347/4120429911.py", line 4, in <cell line: 0>
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py", line 18, in <module>
    from .pi0.configuration_pi0 import PI0Config as PI0Config
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/pi0/__init__.py", line 18, in <module>
    from .modeling_pi0 import PI0Policy
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/pi0/modeling_pi0.py", line 44, in <module>
    from lerobot.policies.pretrained import PreTrainedPolicy, T
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/pretrained.py", line 33, in <module>
    from lerobot.configs.train import TrainPipelineConfig
  File "/tmp/lerobot_v0.4.0/src/lerobot/configs/train.py", line 25, in <module>
    from lerobot import envs
  File "/tmp/lerobot_v0.4.0/src/lerobot/envs/__init__.py", line 15, in <module>
    from .configs import AlohaEnv, EnvConfig, PushtEnv  # no

In [46]:
from pathlib import Path

p = Path("/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py")

print(p)
print("="*80)

with open(p,"r",encoding="utf-8") as f:
    text = f.read()

print(text[:3000])

/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py
# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from .act.configuration_act import ACTConfig as ACTConfig
from .diffusion.configuration_diffusion import DiffusionConfig as DiffusionConfig
from .groot.configuration_groot import GrootConfig as GrootConfig
from .pi0.configuration_pi0 import PI0Config as PI0Config
from .pi05.configuration_pi05 import PI05Config as PI05Config
from .smo

In [47]:
from pathlib import Path

p = Path("/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py")

backup = p.with_suffix(".py.bak")

if not backup.exists():
    backup.write_text(
        p.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

minimal = '''
from .smolvla.configuration_smolvla import SmolVLAConfig
from .smolvla.processor_smolvla import SmolVLANewLineProcessor

__all__ = [
    "SmolVLAConfig",
]
'''

p.write_text(minimal, encoding="utf-8")

print("✅ policies/__init__.py をSmolVLA専用版に変更")

✅ policies/__init__.py をSmolVLA専用版に変更


In [48]:
import sys
import gc
import traceback

# lerobotを再ロード
for mod in list(sys.modules):
    if mod == "lerobot" or mod.startswith("lerobot."):
        del sys.modules[mod]

gc.collect()

if "/tmp/lerobot_v0.4.0/src" not in sys.path:
    sys.path.insert(0, "/tmp/lerobot_v0.4.0/src")

try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from lerobot.configs import PreTrainedConfig

    print("✅ import成功")

except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_2347/703249439.py", line 16, in <cell line: 0>
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py", line 3, in <module>
    from .smolvla.processor_smolvla import SmolVLANewLineProcessor
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/smolvla/processor_smolvla.py", line 23, in <module>
    from lerobot.processor import (
  File "/tmp/lerobot_v0.4.0/src/lerobot/processor/__init__.py", line 43, in <module>
    from .hil_processor import (
  File "/tmp/lerobot_v0.4.0/src/lerobot/processor/hil_processor.py", line 28, in <module>
    from lerobot.teleoperators.teleoperator import Teleoperator
  File "/tmp/lerobot_v0.4.0/src/lerobot/teleoperators/__init__.py", line 18, in <module>
    from .teleoperator import Teleoperator
  File "/tmp/lerobot_v0.4.0/src/lerobot/teleoperators/teleoperator.py", line 22, in <module>
    from lerobot.motors.motors_bus impo

In [49]:
from pathlib import Path

p = Path("/tmp/lerobot_v0.4.0/src/lerobot/processor/__init__.py")

print(p)
print("=" * 80)

print(
    p.read_text(encoding="utf-8")[:4000]
)

/tmp/lerobot_v0.4.0/src/lerobot/processor/__init__.py
#!/usr/bin/env python

# Copyright 2025 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from .batch_processor import AddBatchDimensionProcessorStep
from .converters import (
    batch_to_transition,
    create_transition,
    transition_to_batch,
)
from .core import (
    EnvAction,
    EnvTransition,
    PolicyAction,
    RobotAction,
    RobotObservation,
    TransitionKey,
)
from .delta_action

In [50]:
from pathlib import Path

p = Path(
    "/tmp/lerobot_v0.4.0/src/lerobot/policies/smolvla/processor_smolvla.py"
)

print("=" * 80)
print(p)
print("=" * 80)

text = p.read_text(encoding="utf-8")

# 先頭300行程度
lines = text.splitlines()

for i, line in enumerate(lines[:300], start=1):
    print(f"{i:04d}: {line}")

/tmp/lerobot_v0.4.0/src/lerobot/policies/smolvla/processor_smolvla.py
0001: #!/usr/bin/env python
0002: 
0003: # Copyright 2025 HuggingFace Inc. team. All rights reserved.
0004: #
0005: # Licensed under the Apache License, Version 2.0 (the "License");
0006: # you may not use this file except in compliance with the License.
0007: # You may obtain a copy of the License at
0008: #
0009: #     http://www.apache.org/licenses/LICENSE-2.0
0010: #
0011: # Unless required by applicable law or agreed to in writing, software
0012: # distributed under the License is distributed on an "AS IS" BASIS,
0013: # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
0014: # See the License for the specific language governing permissions and
0015: # limitations under the License.
0016: 
0017: from typing import Any
0018: 
0019: import torch
0020: 
0021: from lerobot.configs.types import PipelineFeatureType, PolicyFeature
0022: from lerobot.policies.smolvla.configuration_smolvla import S

In [51]:
# ============================================
# pyserial を追加してリトライ
# ============================================
import sys
import subprocess
import gc

# pyserial インストール
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyserial"],
    check=True
)
print("✅ pyserial インストール完了")

# lerobot キャッシュを消す
for mod in list(sys.modules):
    if mod == "lerobot" or mod.startswith("lerobot."):
        del sys.modules[mod]

gc.collect()

# policies/__init__.py を元に戻す（PI0が読まれても、pyserialがあれば通る）
from pathlib import Path
p = Path("/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py")
backup = p.with_suffix(".py.bak")
if backup.exists():
    p.write_text(backup.read_text(encoding="utf-8"), encoding="utf-8")
    print("✅ policies/__init__.py を元に戻しました")
else:
    print("⚠️ backup が存在しません、そのまま続行")

# import テスト
import traceback

try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from lerobot.configs import PreTrainedConfig
    print("✅ import成功！")
except Exception:
    traceback.print_exc()

✅ pyserial インストール完了
✅ policies/__init__.py を元に戻しました


Traceback (most recent call last):
  File "/tmp/ipykernel_2347/4094442897.py", line 36, in <cell line: 0>
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/__init__.py", line 17, in <module>
    from .groot.configuration_groot import GrootConfig as GrootConfig
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/groot/__init__.py", line 18, in <module>
    from .modeling_groot import GrootPolicy
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/groot/modeling_groot.py", line 42, in <module>
    from lerobot.policies.groot.groot_n1 import GR00TN15
  File "/tmp/lerobot_v0.4.0/src/lerobot/policies/groot/groot_n1.py", line 176, in <module>
    @dataclass
     ^^^^^^^^^
  File "/usr/lib/python3.12/dataclasses.py", line 1275, in dataclass
    return wrap(cls)
           ^^^^^^^^^
  File "/usr/lib/python3.12/dataclasses.py", line 1265, in wrap
    return _process_class(cls, init, repr, eq, order, unsafe_hash,
           ^^^^^^^

In [ ]:
# ============================================
# v0.6.0学習済み重み → v0.4.0 で読み込みテスト
# ============================================
from pathlib import Path

MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")

print("=== config読み込み ===")
try:
    config = PreTrainedConfig.from_pretrained(MERGED_MODEL_DIR)
    print(f"✅ config読み込み成功")
    print(f"   型: {type(config).__name__}")
    # 主要フィールドを確認
    for attr in ["vlm_model_name", "chunk_size", "n_action_steps"]:
        if hasattr(config, attr):
            print(f"   {attr}: {getattr(config, attr)}")
except Exception as e:
    print(f"❌ config失敗: {type(e).__name__}: {e}")
    raise

print("\n=== model読み込み ===")
try:
    config.device = "cpu"  # メモリ節約のためCPU
    config.use_peft = False
    if hasattr(config, "load_vlm_weights"):
        config.load_vlm_weights = True

    policy = SmolVLAPolicy.from_pretrained(
        MERGED_MODEL_DIR,
        config=config,
        strict=False
    )
    print(f"✅ model読み込み成功！")
    print(f"   型: {type(policy).__name__}")
    print(f"\n🎉 v0.4.0 で v0.6.0 の重みが使えます！")
    print(f"   → submission_v007 の作成に進めます")

    del policy
    gc.collect()

except Exception as e:
    print(f"❌ model失敗: {type(e).__name__}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================
# submission_v007 作成（v0.4.0 + pyserial）
# ============================================
import shutil
import hashlib
import os
from pathlib import Path

CHOSEN_SRC_LEROBOT = Path("/tmp/lerobot_v0.4.0/src/lerobot")
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")
SUBMISSION_DIR = Path("/content/submission")

# クリーンに準備
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 1) lerobot v0.4.0 コピー（元の状態）
DST_LEROBOT = SUBMISSION_DIR / "lerobot"

def ignore_patterns(dir, files):
    return [f for f in files if f == "__pycache__" or f.endswith(".pyc")]

shutil.copytree(CHOSEN_SRC_LEROBOT, DST_LEROBOT, ignore=ignore_patterns)

size = sum(f.stat().st_size for f in DST_LEROBOT.rglob("*") if f.is_file())
print(f"✅ lerobot v0.4.0: {size / 1024 / 1024:.1f} MB")

# 2) __version__.py 確認
version_file = DST_LEROBOT / "__version__.py"
if not version_file.exists():
    version_file.write_text('__version__ = "0.4.0"\n', encoding="utf-8")
    print("✅ __version__.py 自作")

# 3) model_weights コピー
MODEL_WEIGHTS_DIR = SUBMISSION_DIR / "model_weights"
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
for file_path in MERGED_MODEL_DIR.rglob("*"):
    if file_path.is_file():
        target = MODEL_WEIGHTS_DIR / file_path.relative_to(MERGED_MODEL_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, target)
print("✅ model_weights コピー完了")

# 4) policy_server.py 書き出し
policy_server_code = r'''"""ポリシーサーバー v007"""
import argparse
import os
import sys
from abc import ABC, abstractmethod
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

import lerobot
import msgpack
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Request, Response

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs import PreTrainedConfig


class BasePolicy(ABC):
    @abstractmethod
    def get_action(self, obs):
        pass
    @abstractmethod
    def reset(self, instruction=""):
        pass


class MyPolicy(BasePolicy):
    def __init__(self):
        weights_dir = Path(__file__).parent / "model_weights"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        config = PreTrainedConfig.from_pretrained(weights_dir)
        config.device = self.device
        config.use_peft = False
        if hasattr(config, "load_vlm_weights"):
            config.load_vlm_weights = True

        self.policy = SmolVLAPolicy.from_pretrained(
            weights_dir, config=config, strict=False
        )
        self.policy.to(self.device)
        self.policy.eval()
        self.instruction = ""

    def get_action(self, obs):
        with torch.no_grad():
            action_tensor = self.policy.select_action(obs)
            action = action_tensor.cpu().numpy().astype(np.float32).flatten()[:7]
        return action

    def reset(self, instruction=""):
        self.instruction = instruction
        if hasattr(self.policy, "reset"):
            self.policy.reset()


def deserialize_obs(data):
    unpacked = msgpack.unpackb(data, raw=False)
    obs = {}
    for key, val in unpacked.items():
        arr = np.frombuffer(val["data"], dtype=np.dtype(val["dtype"]))
        obs[key] = arr.reshape(val["shape"]).copy()
    return obs

def serialize_action(action):
    return msgpack.packb(
        {"data": action.astype(np.float32).tobytes()},
        use_bin_type=True,
    )

app = FastAPI(title="VLA Policy Server")
_policy = None

def set_policy(policy):
    global _policy
    _policy = policy

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reset")
async def reset_policy(request: Request):
    body = await request.body()
    instruction = ""
    if body:
        import json
        data = json.loads(body)
        instruction = data.get("instruction", "")
    _policy.reset(instruction=instruction)
    return {"status": "ok"}

@app.post("/act")
async def act(request: Request):
    body = await request.body()
    obs = deserialize_obs(body)
    action = _policy.get_action(obs)
    return Response(
        content=serialize_action(action),
        media_type="application/x-msgpack",
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--host", type=str, default="0.0.0.0")
    args = parser.parse_args()
    set_policy(MyPolicy())
    uvicorn.run(app, host=args.host, port=args.port)
'''

(SUBMISSION_DIR / "policy_server.py").write_text(policy_server_code, encoding="utf-8")
print("✅ policy_server.py 書き出し")

# 5) requirements.txt（pyserial 追加）
requirements = """# Policy server dependencies
# ※ プリインストール済みは書かない
# ※ lerobot は同梱するため書かない

# lerobot v0.4.0 の副作用依存
pyserial

# lerobot v0.4.0 のポリシー系固有依存
peft
draccus
num2words

# 保険（プリインストール 4.53.2 と互換ない場合）
transformers==4.57.6
"""

(SUBMISSION_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print("✅ requirements.txt 書き出し（pyserial含む）")

# 6) zip作成
SUBMISSION_ZIP = "/content/submission_v007.zip"
if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
shutil.make_archive("/content/submission_v007", "zip", SUBMISSION_DIR)

size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\n📦 zip サイズ: {size_mb:.1f} MB")

sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

print("\n🎯 次: validate_submission.py で最終検証")